# Swarmie PMF Readiness Index — Colab Calibration

Self-contained backtest box. Serves an open 7B model on the free Colab GPU, runs the swarm over ~200 YC cases, calibrates the index, and downloads `index_weights_v1.json`.

**No paid API.** Everything runs on the Colab T4.

**Before you start:** set **Runtime → Change runtime type → GPU (T4)**. Push your branch with the backtest code to GitHub first (this notebook clones it).

Steps: GPU check → clone → fetch corpus → install → serve vLLM → smoke → full 200 → calibrate → download.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo
Set `REPO_URL` and `BRANCH`. For a private repo, use a fine-grained PAT: `https://<TOKEN>@github.com/<owner>/<repo>.git`.

In [ ]:
REPO_URL = 'https://github.com/hp-8/swarmie.git'  # or https://<PAT>@github.com/hp-8/swarmie.git if private
BRANCH   = 'main'  # the branch holding eval/backtest/ + pmf_index.py

import os
!rm -rf /content/swarmie
!git clone --branch $BRANCH --depth 1 $REPO_URL /content/swarmie
os.chdir('/content/swarmie/backend')
print('cwd:', os.getcwd())
!ls eval/backtest/

## 2b. Fetch the YC corpus
`yc_all.json` is gitignored (a 9.5MB blob), so it isn't in the clone. Pull it fresh from the yc-oss public API into the data dir the runner reads.

In [ ]:
import urllib.request, json
os.makedirs('eval/backtest/data', exist_ok=True)
urllib.request.urlretrieve('https://yc-oss.github.io/api/companies/all.json',
                           'eval/backtest/data/yc_all.json')
print('corpus companies:', len(json.load(open('eval/backtest/data/yc_all.json'))))

## 3. Install dependencies
vLLM (brings its own torch), the backend runtime deps, and the eval-only deps (sklearn/numpy). Takes a few minutes.

In [ ]:
!pip install -q vllm
!pip install -q -r requirements.txt
!pip install -q -r eval/requirements-eval.txt
print('deps installed')

## 4. Serve the model (Qwen2.5-7B-Instruct-AWQ) on localhost:8000
AWQ-quantised 7B fits the T4 (~6GB weights). Launches in the background; the next cell waits until it's ready (first run downloads the weights — a few minutes).

In [ ]:
import subprocess
MODEL = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--quantization', 'awq',
    '--dtype', 'half',
    '--max-model-len', '8192',
    '--gpu-memory-utilization', '0.92',
    '--port', '8000',
], stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
print('vLLM starting, pid', server.pid)

In [ ]:
# Wait until the server answers /v1/models (downloads weights on first run).
import time, urllib.request, json
for i in range(120):
    try:
        r = urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3)
        print('READY:', json.load(r)['data'][0]['id']); break
    except Exception:
        if i % 10 == 0:
            print(f'waiting... {i*5}s'); _=os.system('tail -n 2 /content/vllm.log')
        time.sleep(5)
else:
    print('TIMEOUT — check /content/vllm.log'); _=os.system('tail -n 40 /content/vllm.log')

## 5. Point the pipeline at the local model
All tiers (cheap/deep/synth) fall back to `LLM_MODEL_NAME` when unset, so one served model covers everything. `ROAST_MAX_COST_USD` set high — local inference is free, but the watchdog would otherwise cancel.

In [ ]:
os.environ['LLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['LLM_MODEL_NAME'] = MODEL
os.environ['LLM_API_KEY'] = 'local-vllm'  # vLLM ignores the value but the client requires one
os.environ['SECRET_KEY'] = 'colab-backtest'
os.environ['ROAST_MAX_COST_USD'] = '1000'
print('env set ->', os.environ['LLM_BASE_URL'], MODEL)

## 6. Smoke test (4 cases)
Confirms the whole chain runs end-to-end before the full batch. Synthesis is skipped by default (the index dims don't need it).

In [ ]:
!python -m eval.backtest.runner --limit 4 --n-agents 12 --out /content/smoke.json
import json; d=json.load(open('/content/smoke.json'));
print('rows:', len(d));
print(json.dumps(d[0], indent=2) if d else 'EMPTY — check vllm.log')

## 7. Full run — 200 balanced cases
~200 swarms × ~12 cheap generations, batched by vLLM. Roughly 30–60 min on a T4. Keep this tab active (Colab idles ~90 min).

In [ ]:
!python -m eval.backtest.runner --sample 200 --balanced --repeats 1 --n-agents 20 \
    --out eval/backtest/data/features_tier1.json
import json; d=json.load(open('eval/backtest/data/features_tier1.json'));
print('feature rows:', len(d), '| hits:', sum(r['label']==1 for r in d), '| flops:', sum(r['label']==0 for r in d))

## 8. Calibrate — logistic regression, 5-fold CV AUC
Writes `index_weights_v1.json` and prints the CV AUC. If AUC ∈ [0.45, 0.55] the index labels itself uncalibrated (separation not demonstrated).

In [ ]:
!python -m eval.backtest.calibrate \
    --features eval/backtest/data/features_tier1.json \
    --out app/services/swarm/index_weights_v1.json
import json; w=json.load(open('app/services/swarm/index_weights_v1.json'));
print('\nCV AUC:', round(w['cv_auc'],4), '| n:', w['n'], '| coef:', [round(c,3) for c in w['coef']])

## 9. Download the results
Bring `index_weights_v1.json` (commit it to the repo) and `features_tier1.json` (regeneratable) back to your machine.

In [ ]:
from google.colab import files
files.download('app/services/swarm/index_weights_v1.json')
files.download('eval/backtest/data/features_tier1.json')